In [12]:
import os
import re
import gc
import sqlite3
import unicodedata
import random
import math

import numpy as np
import pandas as pd

from difflib import SequenceMatcher
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

DATA_DIR = "/home/sagemaker-user/Igniters_submission_file"
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

DB_PATH = os.path.join(DATA_DIR, "entity_index.db")

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Pipeline environment ready.")
print("Database:", DB_PATH)

Pipeline environment ready.
Database: /home/sagemaker-user/Igniters_submission_file/entity_index.db


In [13]:
def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower()

    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")

    text = text.replace("&", " and ")

    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_series(series):
    return series.fillna("").astype(str).map(normalize_text)


def first_token(text):
    text = normalize_text(text)

    if not text:
        return ""

    return text.split()[0]


def name_prefix(text, n=4):
    text = normalize_text(text)

    if not text:
        return ""

    return text[:n]


def address_tokens(text):
    text = normalize_text(text)

    if not text:
        return []

    return [
        token for token in text.split()
        if len(token) >= 5
    ]


print("Normalization functions ready.")

Normalization functions ready.


In [4]:
if os.path.exists(DB_PATH):
    print("Existing database found:")
    print(DB_PATH)
    print("If this is from an incomplete previous run, delete it before continuing.")
else:
    print("No existing database. Ready to build.")

Existing database found:
/home/sagemaker-user/Igniters_submission_file/entity_index.db
If this is from an incomplete previous run, delete it before continuing.


In [5]:
conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS entities (
    entity_id TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    business_name TEXT,
    business_address TEXT,
    country TEXT,
    clean_name TEXT,
    clean_address TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS name_prefix_index (
    block_key TEXT,
    entity_id TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS first_token_index (
    block_key TEXT,
    entity_id TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS address_token_index (
    block_key TEXT,
    entity_id TEXT
)
""")

conn.commit()

print("SQLite schema created.")

SQLite schema created.


In [12]:
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_name_prefix
ON name_prefix_index(block_key)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_first_token
ON first_token_index(block_key)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_address_token
ON address_token_index(block_key)
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_entities_id
ON entities(entity_id)
""")

conn.commit()

print("SQLite indexes created.")

SQLite indexes created.


In [10]:
# Cell: Memory-safe chunk processor

CHUNK_SIZE = 25_000   # smaller chunk = lower RAM usage


def process_source_file(filepath, source_name):
    print(f"\nProcessing {source_name}")
    print(f"File: {filepath}")

    total = 0

    for chunk in pd.read_csv(
        filepath,
        sep="\t",
        dtype="string",
        chunksize=CHUNK_SIZE
    ):
        # Fill missing values
        chunk = chunk.fillna("")

        # Normalize name and address
        chunk["clean_name"] = normalize_series(chunk["business_name"])
        chunk["clean_address"] = normalize_series(chunk["business_address"])

        # ---------------------------------------------------------
        # 1. Insert main entity records
        # ---------------------------------------------------------

        records = []

        for row in chunk.itertuples(index=False):

            records.append((
                str(row.entity_id),
                source_name,
                str(row.business_name),
                str(row.business_address),
                str(row.country),
                str(row.clean_name),
                str(row.clean_address)
            ))

        conn.executemany(
            """
            INSERT OR REPLACE INTO entities
            (
                entity_id,
                source,
                business_name,
                business_address,
                country,
                clean_name,
                clean_address
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """,
            records
        )

        # ---------------------------------------------------------
        # 2. Create blocking records
        # ---------------------------------------------------------

        prefix_records = []
        first_records = []
        address_records = []

        for row in chunk.itertuples(index=False):

            entity_id = str(row.entity_id)
            country = normalize_text(row.country)
            name = str(row.clean_name)
            address = str(row.clean_address)

            # -----------------------------------------------------
            # Name prefix block
            # -----------------------------------------------------

            if country and name:

                prefix = name[:4]

                if prefix:
                    prefix_records.append(
                        (country + "|" + prefix, entity_id)
                    )

                # -------------------------------------------------
                # First word block
                # -------------------------------------------------

                name_parts = name.split()

                if name_parts:

                    first = name_parts[0]

                    if len(first) >= 3:
                        first_records.append(
                            (country + "|" + first, entity_id)
                        )

            # -----------------------------------------------------
            # Address token block
            # -----------------------------------------------------

            if country and address:

                tokens = address.split()

                used = set()

                for token in tokens:

                    # Ignore very short tokens
                    if len(token) < 5:
                        continue

                    # Ignore pure numbers
                    if token.isdigit():
                        continue

                    # Avoid duplicate tokens
                    if token in used:
                        continue

                    used.add(token)

                    address_records.append(
                        (country + "|" + token, entity_id)
                    )

                    # Maximum 3 address tokens per business
                    if len(used) >= 3:
                        break

        # ---------------------------------------------------------
        # 3. Insert blocking indexes
        # ---------------------------------------------------------

        if prefix_records:

            conn.executemany(
                """
                INSERT INTO name_prefix_index
                (block_key, entity_id)
                VALUES (?, ?)
                """,
                prefix_records
            )

        if first_records:

            conn.executemany(
                """
                INSERT INTO first_token_index
                (block_key, entity_id)
                VALUES (?, ?)
                """,
                first_records
            )

        if address_records:

            conn.executemany(
                """
                INSERT INTO address_token_index
                (block_key, entity_id)
                VALUES (?, ?)
                """,
                address_records
            )

        # Save this chunk
        conn.commit()

        total += len(chunk)

        # Free memory
        del records
        del prefix_records
        del first_records
        del address_records
        del chunk

        gc.collect()

        # Progress
        if total % 250_000 < CHUNK_SIZE:
            print(f"Processed {total:,} rows")

    print(f"\nFinished {source_name}: {total:,} rows")

In [ ]:
process_source_file(
    os.path.join(DATA_DIR, "train_source2.tsv"),
    "S2"
)


Processing S2
File: /home/sagemaker-user/Igniters_submission_file/train_source2.tsv
Processed 250,000 rows
Processed 500,000 rows
Processed 750,000 rows
Processed 1,000,000 rows
Processed 1,250,000 rows
Processed 1,500,000 rows
